# Calibration pass — summary figures

Renders the figures for `docs/2026-08-19-calibration-pass-findings.md`.

**Kernel: `du`**, which has pandas, seaborn and matplotlib. Not `docparse` — that
environment is deliberately five pure-Python packages and the pipeline depends on
it staying that way.

Everything is derived from the `scores_*.json` reports. Predictions are gitignored
and machine-specific, so an analysis that read them would run only on the machine
that produced them.

Each figure is written as **SVG** for the web and slides, and **PNG** because
pandoc does not embed SVG into `.docx` — it wants EMF, and quietly drops or
rasterises SVG instead.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
sys.path.insert(0, str(REPO))

# The repo is not installed as a package, so the path insert above must
# precede this import.
from analysis import figures  # noqa: E402

figures.apply_style()
FIGURES = REPO / "docs" / "figures"
print(REPO)

## Load

Pass the reports oldest first; a later one wins on a system-name collision, which
is how the superseded prompted runs are replaced by the current ones.

`load` refuses reports that scored different corpora. That is fatal rather than a
warning because a figure mixing two vintages looks entirely normal.

In [ ]:
REPORTS = [
    REPO / "scores_parsers.json",  # MinerU, Docling — read no prompt
    REPO / "scores_v4.json",  # gemma 12B 4-bit, InternVL — current prompt
    REPO / "scores_control.json",  # the BF16 precision controls
    REPO / "scores_31b.json",  # the capacity control
]
available = [p for p in REPORTS if p.exists()]
missing = [p.name for p in REPORTS if not p.exists()]
if missing:
    print(f"not yet scored, skipping: {', '.join(missing)}")

documents = figures.load(*available)
print(f"{len(documents)} document rows across {documents.system.nunique()} systems")
documents.groupby("system").size().sort_index()

## The numbers behind the figures

Medians, not means: a handful of catastrophic pages distort the averages badly,
and quoting means alone misdescribes every system here.

In [ ]:
summary = (
    documents.groupby("system")
    .apply(
        lambda g: pd.Series(
            {
                "median nCER": g.normalised_cer.median(),
                "median sCER": g.strict_cer.median(),
                "gap": (g.strict_cer - g.normalised_cer).median(),
                "amounts correct": g.amounts_correct.sum() / g.truth_amounts.sum(),
                "misfiled": g.misfiled.sum() / g.amounts.sum(),
                "rows aligned": g.aligned.sum() / g.truth_rows.sum(),
                "width ok": g.columns_match.sum() / len(g),
            }
        ),
        include_groups=False,
    )
    .sort_values("median nCER")
)
summary.style.format("{:.4f}")

In [ ]:
# Bank statements alone — the only genuinely hard tables in the corpus. Invoices
# are near-saturated and receipts are a two-column list, so an average over the
# three describes none of them.
statements = documents[documents.doc_type == "bank_statements"]
statements.groupby("system").apply(
    lambda g: pd.Series(
        {
            "amounts": g.truth_amounts.sum(),
            "correct": g.amounts_correct.sum() / g.truth_amounts.sum(),
            "misread": g.misread.sum(),
            "dropped": g.dropped.sum(),
            "invented": g.invented.sum(),
            "pages with an error": (g.misread + g.dropped + g.invented > 0).sum(),
        }
    ),
    include_groups=False,
).sort_values("correct", ascending=False)

## Figures

Each returns the paths it wrote.

In [ ]:
written = figures.all_figures(documents, FIGURES, REPO / "runs_throughput")
for path in written:
    print(path.relative_to(REPO), f"{path.stat().st_size // 1024} KB")

In [ ]:
from IPython.display import SVG, display

for path in sorted(FIGURES.glob("*.svg")):
    display(SVG(filename=str(path)))

## Download

Click to save. SVG for the web and slides; PNG for the `.docx` path.

In [ ]:
from IPython.display import FileLinks

FileLinks(str(FIGURES.relative_to(REPO)))